# **Demo 02: Executing Serial and Parallel Workflows Using CrewAI Agents**


**Objective:** To demonstrate how to build intelligent workflows using CrewAI agents that execute tasks either sequentially (serial) or independently (parallel). This demo highlights how tasks can be structured based on interdependence or concurrency

**Prerequisites:** OpenAI API key, Tavily Key  

**Tools required:** Python

In [1]:
!pip install crewai langchain langchain-openai langchain-community langchain-tavily tavily-python pydantic

  Using cached crewai-1.10.1-py3-none-any.whl.metadata (36 kB)
  Using cached langchain-1.2.10-py3-none-any.whl.metadata (5.7 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_tavily-0.2.17-py3-none-any.whl.metadata (20 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached aiosqlite-0.21.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached chromadb-1.1.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.2 kB)
  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached instructor-1.14.5-py3-none-any.whl.metadata (12 kB)
  Using cached json_repair-0.25.3-py3-none-any.whl.metadata (7.9 kB)
  Using cached json5-0.10.0-py3-none-any.whl.metadata (34 kB)
  Using cached jsonref-1.1.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached lancedb-0.29.2

In [2]:
!pip install litellm

  Using cached fastuuid-0.14.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 20.3 MB/s eta 0:00:00m eta 0:00:010:01:01
Using cached fastuuid-0.14.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (278 kB)


In [ ]:
import requests
from crewai import Agent, Task, Crew
from crewai.llm import LLM
from crewai.tools import BaseTool


In [6]:
!pip install "crewai[azure-ai-inference]"

  Using cached azure_ai_inference-1.0.0b9-py3-none-any.whl.metadata (34 kB)
  Using cached isodate-0.7.2-py3-none-any.whl.metadata (11 kB)
  Using cached azure_core-1.38.2-py3-none-any.whl.metadata (48 kB)
Using cached azure_ai_inference-1.0.0b9-py3-none-any.whl (124 kB)
Using cached azure_core-1.38.2-py3-none-any.whl (217 kB)
Using cached isodate-0.7.2-py3-none-any.whl (22 kB)


In [3]:
# --------------------------------------------------
# Hardcoded credentials
# --------------------------------------------------
AZURE_OPENAI_API_KEY = "2ABecnfxzhRg4M5D6pBKiqxXVhmGB2WvQ0aYKkbTCPsj0JLKsZPfJQQJ99BDAC77bzfXJ3w3AAABACOGi3sC"
AZURE_OPENAI_ENDPOINT = "https://openai-api-management-gw.azure-api.net/"
AZURE_OPENAI_API_VERSION = "2024-12-01-preview"
AZURE_OPENAI_DEPLOYMENT = "gpt-5-mini"

TAVILY_API_KEY = "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"

# --------------------------------------------------
# Tavily Web Search Tool
# --------------------------------------------------
class TavilySearchTool(BaseTool):
    name: str = "web_search"
    description: str = "Search the web for recent information."

    def _run(self, query: str):
        url = "https://api.tavily.com/search"
        payload = {
            "api_key": TAVILY_API_KEY,
            "query": query,
            "max_results": 5
        }

        response = requests.post(url, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()

        results = []
        for r in data.get("results", []):
            title = r.get("title", "No title")
            link = r.get("url", "No URL")
            results.append(f"{title} - {link}")

        return "\n".join(results) if results else "No web results found."


search_tool = TavilySearchTool()

# --------------------------------------------------
# Azure GPT-5-mini LLM via CrewAI LiteLLM path
# --------------------------------------------------
llm = LLM(
    model="azure/gpt-5-mini",
    api_key=AZURE_OPENAI_API_KEY,
    base_url=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
    is_litellm=True,
    temperature=0.3,
)

In [4]:
# --------------------------------------------------
# Agents
# --------------------------------------------------
researcher = Agent(
    role="AI Researcher",
    goal="Find the latest advancements in AI for healthcare",
    backstory=(
        "You are an expert in artificial intelligence and stay updated "
        "with the latest research trends in healthcare."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[search_tool]
)

writer = Agent(
    role="Technical Writer",
    goal="Summarize research into an executive report",
    backstory=(
        "You are an experienced technical writer with expertise in "
        "summarizing healthcare research for executives."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm
)

In [ ]:
# --------------------------------------------------
# SERIAL EXECUTION
# --------------------------------------------------
task_research = Task(
    description=(
        "Use the web search results to explain the top 3 recent advancements in AI for healthcare in 5-6 sentences."
        "Do not call tools again after getting results."
    ),
    expected_output=(
        "Detailed notes on three advancements, with names and explanations."
    ),
    agent=researcher
)

task_write = Task(
    description=(
        "Write a short executive summary using the research notes provided by the AI Researcher. "
        "Limit the answer to about 200 words."
    ),
    expected_output=(
        "An executive summary report of the top 3 AI advancements in healthcare."
    ),
    agent=writer,
    context=[task_research]
)

print("\n=== SERIAL EXECUTION ===")

crew_serial = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    verbose=True
)

serial_result = crew_serial.kickoff()

print("\n[Serial Result]:\n")
try:
    print(serial_result.raw)
except AttributeError:
    print(serial_result)


=== SERIAL EXECUTION ===


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f5fe313f-daf9-41af-9fbd-d3955b313c5d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use the web search results to explain the top 3 recent advancements in AI for healthcare in 5-6          │
│  sentences.Do not call tools again after getting results.                                                       │
│  ID: 70305978-e761-458e-b74e-dfb83e06a233                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Task: Use the web search results to explain the top 3 recent advancements in AI for healthcare in 5-6          │
│  sentences.Do not call tools again after getting results.                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': 'recent advancements AI healthcare 2024 2025 top advancements generative AI foundation models  │
│  clinical decision support drug discovery radiology federated learning explainable AI 2023 2024 2...            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')


[Serial Result]:

Executive Summary — Top 3 AI Advancements in Healthcare

1) Foundation models & generative AI
What: Large language and multimodal models (e.g., GPT-4, Med‑PaLM 2) adapted for clinical decision support, automated chart summarization, patient communication, coding/triage and multimodal interpretation (text, images, signals).
Impact: Major gains in clinician efficiency, faster information access, and improved patient engagement.
Risks/Needs: Hallucinations, bias, and privacy leakage require domain-specific fine‑tuning, rigorous clinical validation, and deployment guardrails (monitoring, human-in-loop workflows, clear liability frameworks).

2) AI‑accelerated drug discovery & protein structure prediction
What: Deep generative models and breakthroughs like AlphaFold enable generative chemistry, structure prediction, and high-throughput in silico screening.
Impact: Shortened timelines and lower early‑stage costs for target identification and lead generation; better insight

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f5fe313f-daf9-41af-9fbd-d3955b313c5d                                                                       │
│  Final Output: Executive Summary — Top 3 AI Advancements in Healthcare                                          │
│                                                                                                                 │
│  1) Foundation models & generative AI                                                                           │
│  What: Large language and multimodal models (e.g., GPT-4, Med‑PaLM 2) adapted for clinical decision support,    │
│  automated chart summarization, patient communication, coding/triage and multimodal interpretation (text,       │
│  images, signals).                                                                                              │
│  Impact: Major gains in clinician efficiency, faster information access, and improved patient engagement.       │
│  Risks/Needs: Hallucinations, bias, and privacy leakage require domain-specific fine‑tuning, rigorous clinical  │
│  validation, and deployment guardrails (monitoring, human-in-loop workflows, clear liability frameworks).       │
│                                                                                                                 │
│  2) AI‑accelerated drug discovery & protein structure prediction                                                │
│  What: Deep generative models and breakthroughs like AlphaFold enable generative chemistry, structure           │
│  prediction, and high-throughput in silico screening.                                                           │
│  Impact: Shortened timelines and lower early‑stage costs for target identification and lead generation; better  │
│  insight into protein conformations.                                                                            │
│  Risks/Needs: Computational outputs require experimental validation and integration with ADMET/pharmacology     │
│  pipelines to de‑risk candidates.                                                                               │
│                                                                                                                 │
│  3) Federated learning & privacy‑preserving ML                                                                  │
│  What: Secure aggregation, differential privacy, and homomorphic encryption enable multi‑site model training    │
│  without centralizing patient data.                                                                             │
│  Impact: Improved model generalizability and fairness across diverse populations; early adoption in radiology,  │
│  pathology, and EHR modeling.                                                                                   │
│  Risks/Needs: Data heterogeneity, communication overhead, and operational complexity necessitate robust         │
│  governance, standards, and infrastructure investment.                                                          │
│                                                                                                                 │
│  Priority recommendations: fund validation and governance frameworks, pilot federated collaborations, and form  │
│  strategic partnerships to operationalize benefits while managing clinical safety and regulatory risk.          │
│                                                                                                                 │
│                                                       

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



╭────────────────────────────── Execution Traces ──────────────────────────────╮
│                                                                              │
│  🔍 Detailed execution traces are available!                                 │
│                                                                              │
│  View insights including:                                                    │
│    • Agent decision-making process                                           │
│    • Task execution flow and timing                                          │
│    • Tool usage details                                                      │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯
Would you like to view your execution traces? [y/N] (20s timeout): 

╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                      

In [ ]:
# --------------------------------------------------
# PARALLEL EXECUTION
# --------------------------------------------------
task_parallel_1 = Task(
    description=(
        "Use web search to list 5 AI companies focusing on drug discovery. "
        "For each company, give one short line about what they specialize in."
    ),
    expected_output="Company names and what they specialize in.",
    async_execution=True,
    agent=researcher
)

task_parallel_2 = Task(
    description=(
        "Write a short report on how AI is transforming patient diagnostics. "
        "Limit the answer to about 200 words."
    ),
    expected_output="A short report with examples and explanation.",
    agent=writer
)

print("\n=== PARALLEL EXECUTION ===")

crew_parallel = Crew(
    agents=[researcher, writer],
    tasks=[task_parallel_1, task_parallel_2],
    verbose=True
)

parallel_result = crew_parallel.kickoff()

print("\n[Parallel Result]:\n")
try:
    print(parallel_result.raw)
except AttributeError:
    print(parallel_result)


=== PARALLEL EXECUTION ===


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0ef75636-25d1-4f95-bc8b-12275bdc7967                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use web search to list 5 AI companies focusing on drug discovery. For each company, give one short line  │
│  about what they specialize in.                                                                                 │
│  ID: f5a2b2e4-eb30-4926-b9a6-a3e0e33aef63                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Task: Use web search to list 5 AI companies focusing on drug discovery. For each company, give one short line  │
│  about what they specialize in.                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': 'Atomwise AI drug discovery company what they do'}                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a short report on how AI is transforming patient diagnostics. Limit the answer to about 200        │
│  words.                                                                                                         │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[Parallel Result]:

AI is reshaping patient diagnostics by enabling earlier, more precise, and scalable detection through pattern recognition, biomarker discovery, and multimodal data integration.

Examples and mechanisms
- Image-based phenomics (Recursion Pharmaceuticals): High‑throughput cellular imaging plus machine learning extracts disease‑specific phenotype signatures, supporting cellular diagnostics, patient stratification, and rapid assessment of therapeutic response.
- Multi‑omics and biomarker discovery (Insilico Medicine): Deep generative models that integrate genomics, proteomics, and metabolomics accelerate identification of diagnostic biomarkers and signatures for complex diseases and aging‑related conditions.
- Knowledge‑graph and literature mining (BenevolentAI): AI links disparate biological data and literature to surface candidate diagnostic markers and mechanistic hypotheses, shortening the path from signal to clinical assay.
- Generative chemistry and structure‑bas

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 0ef75636-25d1-4f95-bc8b-12275bdc7967                                                                       │
│  Final Output: AI is reshaping patient diagnostics by enabling earlier, more precise, and scalable detection    │
│  through pattern recognition, biomarker discovery, and multimodal data integration.                             │
│                                                                                                                 │
│  Examples and mechanisms                                                                                        │
│  - Image-based phenomics (Recursion Pharmaceuticals): High‑throughput cellular imaging plus machine learning    │
│  extracts disease‑specific phenotype signatures, supporting cellular diagnostics, patient stratification, and   │
│  rapid assessment of therapeutic response.                                                                      │
│  - Multi‑omics and biomarker discovery (Insilico Medicine): Deep generative models that integrate genomics,     │
│  proteomics, and metabolomics accelerate identification of diagnostic biomarkers and signatures for complex     │
│  diseases and aging‑related conditions.                                                                         │
│  - Knowledge‑graph and literature mining (BenevolentAI): AI links disparate biological data and literature to   │
│  surface candidate diagnostic markers and mechanistic hypotheses, shortening the path from signal to clinical   │
│  assay.                                                                                                         │
│  - Generative chemistry and structure‑based platforms (Atomwise, Exscientia): Although focused on drug          │
│  discovery, their rapid in‑silico screening and design methods enable discovery of molecular probes and         │
│  companion diagnostics that validate disease targets and improve diagnostic specificity.                        │
│                                                                                                                 │
│  Impact                                                                                                         │
│  AI delivers faster discovery of diagnostic signatures, enhanced sensitivity/specificity through multimodal     │
│  fusion, and scalable screening for companion diagnostics—supporting personalized care and trial enrichment.    │
│                                                                                                                 │
│  Challenges                                                                                                     │
│  Clinical validation, regulatory approval, explainability, data heterogeneity, and integration into clinical    │
│  workflows remain critical barriers. Priority actions include prospective validation studies, standardized      │
│  pipelines, and clinician‑centric implementation to realize operational benefit.                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



╭────────────────────────────── Execution Traces ──────────────────────────────╮
│                                                                              │
│  🔍 Detailed execution traces are available!                                 │
│                                                                              │
│  View insights including:                                                    │
│    • Agent decision-making process                                           │
│    • Task execution flow and timing                                          │
│    • Tool usage details                                                      │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯
Would you like to view your execution traces? [y/N] (20s timeout): 

╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                      